# TheLook Lakehouse — Stream Processor

Pipeline: **PostgreSQL → Debezium → Kafka → Spark Structured Streaming → Delta Lake (MinIO)**

Run cells in order:
1. Health checks — wait for all services
2. Simulator — generate CDC data into PostgreSQL
3. Spark Streaming — read Kafka, write Delta Lake

## 1 — Health Checks

In [ ]:
import socket, time

def wait_for_tcp(host, port, label, timeout=120):
    print(f"Waiting for {label} ({host}:{port})...", end="", flush=True)
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            socket.create_connection((host, port), timeout=2).close()
            print(" OK"); return
        except OSError:
            print(".", end="", flush=True); time.sleep(3)
    raise TimeoutError(f"{label} not ready after {timeout}s")

wait_for_tcp("postgres",       5432,  "PostgreSQL")
wait_for_tcp("minio",          9000,  "MinIO")
wait_for_tcp("hive-metastore", 9083,  "Hive Metastore")
wait_for_tcp("spark-master",   7077,  "Spark Master")
wait_for_tcp("broker",         29092, "Kafka Broker")

print("\nAll services ready.")

## 2 — Simulator

In [ ]:
import subprocess, sys, os

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "faker", "psycopg2-binary", "SQLAlchemy"],
    check=True
)

simulator = subprocess.Popen(
    [
        sys.executable, "simulator/generator.py",
        "--db-host",     "postgres",
        "--db-user",     os.getenv("POSTGRES_USER",     "admin"),
        "--db-password", os.getenv("POSTGRES_PASSWORD", "admin123"),
        "--db-name",     os.getenv("POSTGRES_DB",       "tpcds"),
        "--db-schema",   "public",
        "--init-num-users", "1000",
        "--avg-qps",     "2",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    cwd="/home/jovyan/work",
)

# Show first few lines of output to confirm it started
for _ in range(10):
    line = simulator.stdout.readline().decode()
    if line: print(line, end="")

print(f"\nSimulator running (PID {simulator.pid})")

In [ ]:
# Wait for Kafka topics (created by Debezium after first CDC event)
from kafka import KafkaAdminClient

REQUIRED_TOPICS = [
    "tpcds.public.users", "tpcds.public.orders", "tpcds.public.order_items",
    "tpcds.public.events", "tpcds.public.products", "tpcds.public.dist_centers",
]

print("Waiting for Kafka topics...", end="", flush=True)
deadline = time.time() + 120
while time.time() < deadline:
    try:
        admin = KafkaAdminClient(bootstrap_servers="broker:29092")
        existing = set(admin.list_topics()); admin.close()
        if all(t in existing for t in REQUIRED_TOPICS):
            print(" OK"); break
    except Exception:
        pass
    print(".", end="", flush=True); time.sleep(5)
else:
    raise TimeoutError("Kafka topics not ready. Is the Debezium connector registered?")

## 3 — Spark Streaming

In [ ]:
from operators.streaming import SparkStreaming
from utils.config import CHECKPOINT_BASE, DELTA_BASE
from utils.schemas import TOPIC_CONFIG
from utils.streaming_functions import parse_debezium_event

spark = SparkStreaming.get_instance()

for topic, (schema, table_name) in TOPIC_CONFIG.items():
    raw = SparkStreaming.create_kafka_read_stream(spark, topic)
    parsed = parse_debezium_event(raw, schema)
    SparkStreaming.create_delta_write_stream(parsed, table_name, CHECKPOINT_BASE, DELTA_BASE)
    print(f"Stream started: {topic} -> {DELTA_BASE}/{table_name}")

SparkStreaming.register_delta_tables(spark, TOPIC_CONFIG, DELTA_BASE)
print("\nAll streams running. Interrupt kernel to stop.")

In [ ]:
# Keep streams alive — interrupt kernel (■) to stop
try:
    spark.streams.awaitAnyTermination()
finally:
    simulator.terminate()
    print("Simulator stopped.")